In [1]:
import sys # aby importy z dołu działały
sys.path.append(r"C:\Users\Admin\OneDrive\____Moje-MOJE\MyProjects_4Fun\projects\World of Warcraft\python-etl")

In [ ]:
from scraper_wiki_main import parsuj_misje_z_url, dekompresuj_html
from moduly.services_persist_wynik import normalizuj_nazwe_npc
from moduly.db_core import utworz_engine_do_db
from sqlalchemy import text
import json

In [3]:
silnik = utworz_engine_do_db()

In [4]:
q_select_id_puste_npc = text("""
    SELECT MISJA_ID_MOJE_PK
    FROM dbo.MISJE
    WHERE 1=1
      AND (NPC_START_ID IS NULL OR NPC_KONIEC_ID IS NULL)
      AND MISJA_ID_Z_GRY <> 123456789
""")

In [5]:
q_select_npc = text("""
    SELECT NPC_ID_MOJE_PK
    FROM dbo.NPC
    WHERE NAZWA = :npc
""")

In [6]:
q_select_html = text("""
WITH A AS (
	SELECT 
		MISJA_ID_MOJE_FK, 
		HTML_SKOMPRESOWANY,
		ROW_NUMBER() OVER (PARTITION BY MISJA_ID_MOJE_FK ORDER BY DATA_WYSCRAPOWANIA DESC) AS RNK
	FROM dbo.ZRODLO
	WHERE MISJA_ID_MOJE_FK = :m
)

SELECT HTML_SKOMPRESOWANY
FROM A
WHERE RNK = 1
;
""")

In [7]:
q_update_npc = text("""
    UPDATE dbo.MISJE
    SET NPC_START_ID = COALESCE(NPC_START_ID, :id_startnpc),
        NPC_KONIEC_ID = COALESCE(NPC_KONIEC_ID, :id_koniecnpc)
    WHERE MISJA_ID_MOJE_PK = :id_misji
""")

In [8]:
with silnik.begin() as conn:
    #w = conn.execute(q_select_npc, {"npc": start_NPC}).scalar_one()
    misje_puste_npc = conn.execute(q_select_id_puste_npc).scalars().all()
    for id_misji in misje_puste_npc:
        try:
            html_skompresowany = conn.execute(q_select_html, {"m": id_misji}).scalar_one()
            html = dekompresuj_html(html_skompresowany)
            wynik = parsuj_misje_z_url(None, html_content=html)
            #print(wynik)
            start_NPC = normalizuj_nazwe_npc(wynik["Misje_EN"]["Podsumowanie_EN"].get("Start_NPC"))
            koniec_NPC = normalizuj_nazwe_npc(wynik["Misje_EN"]["Podsumowanie_EN"].get("Koniec_NPC"))
            #print(start_NPC,koniec_NPC)
            id_startnpc = conn.execute(q_select_npc, {"npc": start_NPC}).scalar_one_or_none()
            id_koniecnpc = conn.execute(q_select_npc, {"npc": koniec_NPC}).scalar_one_or_none()
            #print(id_misji, "----------", id_startnpc, id_koniecnpc)
            conn.execute(q_update_npc, {
                "id_startnpc": id_startnpc,
                "id_koniecnpc": id_koniecnpc,
                "id_misji": id_misji
            })
        except Exception as e:
            print(f"Misja: {id_misji} sie wysypala, blad: {e}")